In [ ]:
import matplotlib.pylab as plt
import xarray as xr
import pint_xarray
import numpy as np
import cftime
from functools import partial

from pism_terra.processing import integrate_rate, preprocess_netcdf

ref_year = "1985"

In [ ]:
ds = xr.open_mfdataset("/Users/andy/base/pism-terra/2026_08_ismip7_calib_vc/output/scalar/scalar_g900m_id_CESM2-WACCM_uq_*_none_1985-01-01_2015-01-01.nc", 
                       preprocess=partial(preprocess_netcdf, uq_regexp='uq_(.+?)_', drop_dims=["nv"], drop_vars=["time_bounds"]), 
                       join="outer")

In [ ]:
ds = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_historical/output/scalar/basin_g900m_id_CESM2-WACCM_none_1990-01-01_2015-01-01.nc")
ds = ds.expand_dims({"uq_id": ["free"]}).sel(basin="GIS")

In [ ]:
ds = ds.convert_calendar("standard", use_cftime=False).resample(time='MS').mean('time').pint.quantify()

In [ ]:
grace = xr.open_dataset("/Users/andy/base/pism-ragis/data/grace/greenland_mass_balance.nc").squeeze().pint.quantify()
mankoff = xr.open_dataset("/Users/andy/base/pism-ragis/data/mass_balance/mankoff_greenland_mass_balance_clean.nc").pint.quantify()
mankoff = mankoff.sum(dim="region").resample(time='MS').mean('time').pint.to("Gt/yr")

sigma = 2
mankoff_mb = mankoff.MB
mankoff_cmb = integrate_rate(mankoff.MB)
mankoff_cmb = mankoff_cmb - mankoff_cmb.sel(time=ref_year, method="nearest")
mankoff_mb_err = mankoff.MB_err
mankoff_smb = mankoff.SMB
mankoff_glf = -mankoff.D

In [ ]:
mass = ds.ice_mass_glacierized
#mass = integrate_rate(ds.tendency_of_ice_mass_due_to_surface_mass_flux) + integrate_rate(ds.grounding_line_flux)
mass = mass - mass.sel(time=ref_year, method="nearest")
mass = mass.pint.to("Gt")
#mass = (ds.tendency_of_ice_mass.pint.to("Gt/yr") - xr.DataArray(400).pint.quantify("Gt/yr")).cumsum(dim="time") 
#mass = mass - mass.sel(time="2002", method="nearest")

fig, ax = plt.subplots(1, 1)
mass.plot(hue="uq_id", ax=ax, color="0.5", lw=0.5, add_legend=False)
grace.cumulative_mass_balance.plot(ax=ax, color="#DC267F")
mankoff_cmb.plot(ax=ax, lw=2, color="#FE6100")
ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))


In [ ]:
#glf = ds.tendency_of_ice_mass_due_to_discharge
glf = ds.grounding_line_flux
smb = ds.tendency_of_ice_mass_due_to_surface_mass_flux
mb = smb + glf 

uq_vals = mb["uq_id"].values
palette = dict(zip(uq_vals, plt.cm.Paired(np.linspace(0, 1, len(uq_vals)))))

fig, axs = plt.subplots(3, 1, figsize=(6.4, 10.4))
mankoff_mb.resample(time='YS').mean('time').plot(ax=axs[0], color="#FE6100", lw=2)
mankoff_smb.resample(time='YS').mean('time').plot(ax=axs[1], color="#FE6100", lw=2)
mankoff_glf.resample(time='YS').mean('time').plot(ax=axs[2], color="#FE6100", lw=2)
for k, (da, ls) in enumerate([(mb, "solid"), (smb, "dotted"), (glf, "dashed")]):
    ax = axs[k]
    # for u in da["uq_id"].values:
    #     #da.sel(uq=u).plot(ax=ax, color=palette[u], ls=ls)
    #     da.sel(uq_id=u).plot(ax=ax, color=palette[u], ls="solid", lw=2)
    da.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=0.5, color="0.5", add_legend=False)        
    ax.set_title(None)
    ax.axhline(0, color="k", lw=0.5, ls="dotted")
    ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))
    ax.set_ylim(-1000, 1000)

axs[0].fill_between(mankoff_mb - sigma * mankoff_mb_err, mankoff_mb + sigma * mankoff_mb_err, alpha=0.5, color="r")


In [ ]:
ds.ice_mass_glacierized.sel(uq="GrIS")

In [ ]:
ax.fill_between(mankoff_mb - sigma * mankoff_mb_err, mankoff_mb + sigma * mankoff_mb_err, alpha=0.5, color="r")

In [ ]:
ds

In [ ]:
ds_basin.ice_mass_glacierized


In [ ]:
dt_ns = (mankoff["time"].diff("time") / np.timedelta64(1, "ns")) 

In [ ]:
da = xr.DataArray(1).pint.quantify("m^2 yr^-1")

In [ ]:
da.pint.dimensionality

In [ ]:
dt_ns

In [ ]:
integrate_rate(ds.tendency_of_ice_mass_due_to_surface_mass_flux)

In [ ]:
ds.tendency_of_ice_mass_due_to_surface_mass_flux